# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

- Load your committed dataset.
- Apply the feature engineering, preprocessing, and train/val/test split
  consistent with the business framing you wrote in `REPORT.md`.
- Train the same model — linear regression for a regression target,
  logistic regression for a binary one — three ways, on the same split:
  scikit-learn, a from-scratch PyTorch loop (Session 5's manual approach:
  raw tensors, autograd, a manual gradient-step update), and the standard
  `torch.nn.Module` + `torch.optim` workflow.
- Evaluate all three against each other and against a naive baseline.
- Save whatever `app.py`/`app.ipynb` needs to build its dashboard without
  retraining anything.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

In [ ]:
import pandas as pd
df = pd.read_csv("data/bank.csv", sep=";")
df.head()

## Explore categorical values and previous-contact history

Before choosing model inputs, inspect how categorical values relate to the target. These rates are descriptive, not causal, and small groups can have noisy rates. The previous-contact summary distinguishes customers with `pdays = -1` (no recorded previous contact) from the rest. This evidence will inform feature selection for the agreed decision point: before a new campaign starts.

In [ ]:
cat_cols = [
    col for col in df.select_dtypes(include=["object", "str"]).columns
    if col != "y"
]

categorical_summaries = {
    col: (
        df.groupby(col, dropna=False)["y"]
        .agg(
            n="size",
            yes_rate=lambda values: values.eq("yes").mean(),
        )
        .sort_values("n", ascending=False)
    )
    for col in cat_cols
}

contact_history = df.groupby(
    df["pdays"].eq(-1).map({True: "never_contacted", False: "contacted_before"})
)["y"].agg(
    n="size",
    yes_rate=lambda values: values.eq("yes").mean(),
)
contact_history

## Choose available features and split the data

We will use customer profile/financial information and prior-campaign history. We leave out `duration` and `campaign`, which describe the campaign being predicted, and leave out `contact`, `day`, and `month` because they describe how a historical campaign was carried out rather than stable pre-campaign customer information. The special value `pdays = -1` means there was no previous contact, not negative one day: create an indicator and set the numeric days-since-contact value to zero for those rows.

Keep a stratified test set aside for the final comparison, and use validation data for the dashboard's threshold exploration.

In [ ]:
from sklearn.model_selection import train_test_split

y = df["y"].eq("yes").astype("int64")
X = df[
    [
        "age", "job", "marital", "education", "default", "balance",
        "housing", "loan", "pdays", "previous", "poutcome",
    ]
].copy()

X["was_previously_contacted"] = X["pdays"].ne(-1).astype("int64")
X["days_since_previous_contact"] = X["pdays"].where(X["pdays"].ne(-1), 0)
X = X.drop(columns="pdays")

train_val_idx, test_idx = train_test_split(
    df.index, test_size=0.20, random_state=42, stratify=y
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.25, random_state=42, stratify=y.loc[train_val_idx]
)

X_train, X_val, X_test = X.loc[train_idx], X.loc[val_idx], X.loc[test_idx]
y_train, y_val, y_test = y.loc[train_idx], y.loc[val_idx], y.loc[test_idx]

split_sizes = {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)}

## Preprocess once, fit on training data only

Logistic regression needs numeric inputs. Standardize numeric columns so their scales are comparable, and one-hot encode categories without inventing an ordering. Fit these transformations on the training set only, then reuse them for validation and test data to avoid data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_COLUMNS = [
    "age", "balance", "previous", "was_previously_contacted",
    "days_since_previous_contact",
]
CATEGORICAL_COLUMNS = [
    "job", "marital", "education", "default", "housing", "loan", "poutcome",
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), NUMERIC_COLUMNS),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLUMNS),
    ]
)

X_train_ready = preprocessor.fit_transform(X_train).astype("float64")
X_val_ready = preprocessor.transform(X_val).astype("float64")
X_test_ready = preprocessor.transform(X_test).astype("float64")
feature_names = preprocessor.get_feature_names_out()

input_size = X_train_ready.shape[1]

## Train logistic regression three ways

All methods use the same training rows, transformed features, and L2 regularization. The PyTorch models have one linear layer and a sigmoid probability output; the manual version updates weights directly using gradients, while the standard version uses `nn.Linear` and `torch.optim.SGD`. The same regularization strength and learning rate keep the comparison focused on implementation.

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.linear_model import LogisticRegression

REGULARIZATION_C = 1.0
LEARNING_RATE = 0.1
EPOCHS = 10000
l2_strength = 1.0 / (REGULARIZATION_C * len(y_train))
y_train_array = y_train.to_numpy(dtype=np.float64).reshape(-1, 1)

# scikit-learn logistic regression
sklearn_model = LogisticRegression(
    C=REGULARIZATION_C, solver="lbfgs", max_iter=2000, tol=1e-10
)
sklearn_model.fit(X_train_ready, y_train_array.ravel())

# Manual PyTorch: tensors, autograd, and an explicit gradient update
X_train_tensor = torch.tensor(X_train_ready, dtype=torch.float64)
y_train_tensor = torch.tensor(y_train_array, dtype=torch.float64)
manual_weights = torch.zeros(
    (X_train_ready.shape[1], 1), dtype=torch.float64, requires_grad=True
)
manual_bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

for epoch in range(EPOCHS):
    logits = X_train_tensor @ manual_weights + manual_bias
    data_loss = F.binary_cross_entropy_with_logits(logits, y_train_tensor)
    loss = data_loss + 0.5 * l2_strength * manual_weights.square().sum()
    loss.backward()
    with torch.no_grad():
        manual_weights -= LEARNING_RATE * manual_weights.grad
        manual_bias -= LEARNING_RATE * manual_bias.grad
        manual_weights.grad.zero_()
        manual_bias.grad.zero_()

# Standard PyTorch: nn.Module and torch.optim
class TorchLogisticRegression(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.linear = nn.Linear(input_size, 1, dtype=torch.float64)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, features):
        return self.linear(features)

standard_model = TorchLogisticRegression(X_train_ready.shape[1])
optimizer = torch.optim.SGD(
    [
        {"params": standard_model.linear.weight, "weight_decay": l2_strength},
        {"params": standard_model.linear.bias, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
)

for epoch in range(EPOCHS):
    logits = standard_model(X_train_tensor)
    loss = F.binary_cross_entropy_with_logits(logits, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

## Compare on the held-out test set and save artifacts

The baseline always predicts the training-set majority class (`no`). Alongside accuracy, report precision, recall, F1, ROC-AUC, and average precision; this makes the minority `yes` class visible. Test predictions are for final comparison, while the dashboard's threshold slider uses validation predictions. Persist models and the fitted preprocessor so the app only loads and uses trained artifacts.

In [ ]:
from pathlib import Path
import joblib
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score, precision_score,
    recall_score, roc_auc_score,
)

MODEL_NAMES = ["scikit-learn", "manual PyTorch", "standard PyTorch"]

def probabilities_for_torch(features):
    feature_tensor = torch.tensor(features, dtype=torch.float64)
    with torch.no_grad():
        manual_logits = feature_tensor @ manual_weights + manual_bias
        standard_logits = standard_model(feature_tensor)
        return {
            "manual PyTorch": torch.sigmoid(manual_logits).numpy().ravel(),
            "standard PyTorch": torch.sigmoid(standard_logits).numpy().ravel(),
        }

def model_probabilities(features):
    predictions = {"scikit-learn": sklearn_model.predict_proba(features)[:, 1]}
    predictions.update(probabilities_for_torch(features))
    return predictions

validation_probabilities = model_probabilities(X_val_ready)
test_probabilities = model_probabilities(X_test_ready)

def classification_metrics(actual, probabilities):
    predicted = (probabilities >= 0.5).astype("int64")
    return {
        "accuracy": accuracy_score(actual, predicted),
        "precision_yes": precision_score(actual, predicted, zero_division=0),
        "recall_yes": recall_score(actual, predicted, zero_division=0),
        "f1_yes": f1_score(actual, predicted, zero_division=0),
        "roc_auc": roc_auc_score(actual, probabilities),
        "average_precision": average_precision_score(actual, probabilities),
    }

test_actual = y_test.to_numpy(dtype="int64")
metric_rows = {
    name: classification_metrics(test_actual, test_probabilities[name])
    for name in MODEL_NAMES
}
majority_probability = float(y_train.mean())
metric_rows["naive majority baseline"] = classification_metrics(
    test_actual, np.full(len(test_actual), majority_probability)
)
test_metrics = pd.DataFrame.from_dict(metric_rows, orient="index")

models_dir = Path("models")
models_dir.mkdir(exist_ok=True)
torch.save(
    {"weights": manual_weights.detach(), "bias": manual_bias.detach()},
    models_dir / "manual_pytorch.pt",
)
torch.save(standard_model.state_dict(), models_dir / "standard_pytorch.pt")

artifacts = {
    "preprocessor": preprocessor,
    "sklearn_model": sklearn_model,
    "input_size": X_train_ready.shape[1],
    "feature_names": feature_names,
    "test_features": X_test.reset_index(drop=True),
    "test_features_ready": X_test_ready,
    "test_actual": test_actual,
    "test_probabilities": test_probabilities,
    "validation_features": X_val.reset_index(drop=True),
    "validation_features_ready": X_val_ready,
    "validation_actual": y_val.to_numpy(dtype="int64"),
    "validation_probabilities": validation_probabilities,
    "test_metrics": test_metrics,
    "target_distribution": df["y"].value_counts().to_dict(),
    "model_names": MODEL_NAMES,
}
joblib.dump(artifacts, models_dir / "bank_marketing_artifacts.joblib")